# Exercise 4: RAG by hand

The documents: 8 EMA medicine summaries, and Narvelia, a medicine that does not exist.

1. Search: find the passages closest to a question
2. Answer: put the passages in the prompt
3. A question the documents do not answer

Option, if you finish early: does the passage size matter?

Run the cells one by one, from the top: **Shift+Enter**. After you change a cell, run it again, then the cells below it.

In [ ]:
import pandas as pd
from IPython.display import Markdown

import checks
from course import ask, embed, rag_index, similarity, test_questions

## 1 · Search

> **The idea:** the search compares the meaning of the question with the meaning of each passage, with embeddings (slide "Cosine similarity")

The documents are cut into passages of 150 words, and each passage becomes a vector of 384 numbers.
This cell runs once, on your laptop: nothing is sent.

In [ ]:
chunks, vectors = rag_index(150)
Markdown(f"{len(chunks)} passages · matrix of {vectors.shape[0]} × {vectors.shape[1]} numbers")

Two texts to compare: a question and a sentence.

In [ ]:
question = "How often is Mounjaro injected?"
sentence = "Mounjaro is injected once a week."

`embed` turns a text into a vector of 384 numbers. The first 5:

In [ ]:
embed(question)[:5]

**Write one line**: the similarity between `question` and `sentence`. `similarity(a, b)` compares two vectors.

In [ ]:
score = ...  # TODO

In [ ]:
checks.ex4_score(score, question, sentence)

<details><summary>💡 Hint</summary>

`similarity` takes two vectors: `embed(...)` for each text.

</details>

In [ ]:
if score is ...:
    result = Markdown("First write the line above.")
else:
    result = Markdown(f"""
**Score: {score:.2f}.** 1: the same meaning. 0: no relation.

**Change the sentence.** Target: a score near 0 (another topic), then a high score with different words (same meaning).
""")
result

The search does the same with each passage, then keeps the 3 best scores.

In [ ]:
question_vector = embed(question)
scores = [similarity(question_vector, vector) for vector in vectors]
best = sorted(range(len(chunks)), key=lambda index: scores[index], reverse=True)[:3]
pd.DataFrame([
    {"score": round(float(scores[index]), 2), "passage": chunks[index]["id"], "text": chunks[index]["text"][:200] + " …"}
    for index in best
])

- The search runs on your laptop, in less than one second
- The passage closest to the question is not always the one that contains the answer

**Change the question.** Target: find a question where the best passage comes from the wrong medicine.
Try "Which medicines treat chronic kidney disease?": two documents contain the answer.

## 2 · Answer

> **The idea:** the model does not know your documents. The program puts the best passages in the prompt (slide "The answer prompt")

**Complete the instructions**: three rules. Use only the passages. Cite the passage id. If the answer is not there, say so.

In [ ]:
instructions = "Answer the question."

In [ ]:
checks.ex4_instructions(instructions)

The program builds the prompt from your instructions, the 3 best passages and the question, then sends it to the API.

In [ ]:
context = "\n\n".join(f"[{chunks[index]['id']}]\n{chunks[index]['text']}" for index in best)
prompt = f"{instructions}\n\n<passages>\n{context}\n</passages>\n\nQuestion: {question}"
answer, seconds = ask(prompt, "api", temperature=0)
Markdown(f"{answer}\n\n*{seconds:.1f} s, Groq*")

To see the prompt sent to the model: `print(prompt)` in a new cell.

- The model reads only the 3 passages: the search decides what it can know
- With the ids, you can check each fact in the document

<details><summary>Question</summary>

The answer is wrong. Where do you look first: the model or the passages?

</details>

## 3 · A question the documents do not answer

> **The idea:** with no useful passage, a good prompt makes the model say that it does not know, instead of making up an answer (slide "Against hallucination")

**Change the question** in part 1, then run the cells again from there. For example: "How much does Wegovy cost in Belgium?", or a medicine that is not in the documents.

<details><summary>Question</summary>

Does the model say that the answer is not in the documents? Remove your third rule and try again.

</details>

---
## Option · Does the passage size matter?

For each test question, the expected document is known. *hit@3*: the share of questions where it is in the 3 best passages.
**Change the size** (50 to 500 words). Target: find a size where hit@3 goes down.

In [ ]:
size = 150

In [ ]:
sized_chunks, sized_vectors = rag_index(size)
expected = [item for item in test_questions() if item["in_corpus"]]
hits = sum(
    any(sized_chunks[int(index)]["doc"] in item["sources"] for index in (sized_vectors @ embed(item["question"])).argsort()[::-1][:3])
    for item in expected
)
Markdown(f"**hit@3 = {hits} / {len(expected)}** with {len(sized_chunks)} passages of {size} words")